In [13]:
from datasets import load_dataset

ds = load_dataset("yaful/MAGE", split="validation")

In [14]:
print(ds)

Dataset({
    features: ['text', 'label', 'src'],
    num_rows: 56792
})


In [15]:
ds.features

{'text': Value(dtype='string', id=None),
 'label': Value(dtype='int64', id=None),
 'src': Value(dtype='string', id=None)}

In [16]:
from collections import Counter
import math

import matplotlib.pyplot as plt
import pandas as pd
import spacy

# Keep the same verb set as the main pipeline so the notebook matches the analysis.
fancy_verbs = [
    "delve",
    "underscore",
    "showcase",
    "enhance",
    "exhibit",
    "garner",
    "align",
    "foster",
]

nlp = spacy.load("en_core_web_sm", disable=["ner", "textcat"])


def first_answer(values) -> str:
    return values[0] if isinstance(values, list) and values else ""


def dependency_entropy(text: str, nlp) -> float:
    doc = nlp(str(text))
    deps = [token.dep_ for token in doc if token.dep_ != "punct"]
    if not deps:
        return 0.0
    counts = Counter(deps)
    total = sum(counts.values())
    return -sum((count / total) * math.log(count / total) for count in counts.values())


def count_fancy_verbs(text: str, nlp, verbs: list[str]) -> dict[str, int]:
    doc = nlp(str(text))
    lemmas = [token.lemma_.lower() for token in doc]
    counts = {}
    for verb in verbs:
        counts[verb] = lemmas.count(verb)
    return counts


# Build one dataframe with the relevant fields.
mage = pd.DataFrame({
    "text": ds["text"],
    "label": ds["label"],
    "src": ds["src"],
})
mage["word_count"] = mage["text"].astype(str).str.split().str.len()
mage["dependency_entropy"] = [dependency_entropy(text, nlp) for text in mage["text"]]

verb_counts = pd.DataFrame(
    [count_fancy_verbs(text, nlp, fancy_verbs) for text in mage["text"]],
    index=mage.index,
)
verb_counts.columns = [f"verb_{verb}" for verb in verb_counts.columns]
mage = pd.concat([mage, verb_counts], axis=1)
mage["fancy_verb_count"] = mage[[f"verb_{verb}" for verb in fancy_verbs]].sum(axis=1)
mage["fancy_verb_per_1k_words"] = mage["fancy_verb_count"] / mage["word_count"].replace(0, pd.NA) * 1000.0

# Collapse sources into human vs AI so we can sample a balanced subset.
mage["group"] = mage["src"].where(mage["src"].str.contains("human", na=False), "ai")
mage["group"] = mage["group"].mask(mage["group"] != "ai", "human")

sample_size = 10
human_sample = mage[mage["group"] == "human"].sample(n=min(sample_size, (mage["group"] == "human").sum()), random_state=42)
ai_sample = mage[mage["group"] == "ai"].sample(n=min(sample_size, (mage["group"] == "ai").sum()), random_state=42)
sampled = pd.concat([human_sample, ai_sample], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("Sample sizes:")
print(sampled["group"].value_counts())
print("\nSource breakdown inside the sample:")
print(sampled.groupby(["group", "src"]).size().sort_values(ascending=False))

# Summary by group.
group_summary = (
    sampled.groupby("group")
    .agg(
        n=("fancy_verb_count", "size"),
        mean_fancy_verb_count=("fancy_verb_count", "mean"),
        median_fancy_verb_count=("fancy_verb_count", "median"),
        mean_fancy_verb_per_1k_words=("fancy_verb_per_1k_words", "mean"),
        mean_entropy=("dependency_entropy", "mean"),
        mean_word_count=("word_count", "mean"),
    )
    .reset_index()
    .sort_values("mean_fancy_verb_per_1k_words", ascending=False)
)
print("\nGroup summary")
print(group_summary)

# Per-source breakdown within the sampled subset.
source_summary = (
    sampled.groupby("src")
    .agg(
        n=("fancy_verb_count", "size"),
        mean_fancy_verb_count=("fancy_verb_count", "mean"),
        median_fancy_verb_count=("fancy_verb_count", "median"),
        mean_fancy_verb_per_1k_words=("fancy_verb_per_1k_words", "mean"),
        mean_entropy=("dependency_entropy", "mean"),
        mean_word_count=("word_count", "mean"),
    )
    .reset_index()
    .sort_values("mean_fancy_verb_per_1k_words", ascending=False)
)
print("\nSource summary within the balanced sample")
print(source_summary)

# Per-verb breakdown by group, normalized by document length.
per_verb_group = (
    sampled.groupby("group")[[f"verb_{verb}" for verb in fancy_verbs]]
    .mean()
    .sort_index()
)
print("\nPer-verb mean counts by group")
print(per_verb_group)

# Plot the group-level comparison and the per-verb pattern.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

group_order = group_summary["group"].tolist()
axes[0].boxplot(
    [sampled.loc[sampled["group"] == group, "fancy_verb_per_1k_words"].dropna() for group in group_order],
    labels=group_order,
    showmeans=True,
)
axes[0].set_title("Fancy verbs per 1k words by group")
axes[0].set_ylabel("Fancy verbs per 1k words")
axes[0].grid(alpha=0.25, axis="y")

im = axes[1].imshow(per_verb_group.values, aspect="auto", cmap="Blues")
axes[1].set_xticks(range(len(fancy_verbs)))
axes[1].set_xticklabels(fancy_verbs, rotation=30, ha="right")
axes[1].set_yticks(range(len(per_verb_group.index)))
axes[1].set_yticklabels(per_verb_group.index)
axes[1].set_title("Per-verb mean counts by group")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("\nInterpretation check:")
print("If the AI group uses the fancy verbs more often than the human group, that gives you the lexical signal you want in a balanced 1000/1000 sample.")


KeyboardInterrupt: 

In [ ]:
from collections import Counter
import math

import matplotlib.pyplot as plt
import pandas as pd
import spacy

# Keep the same verb set as the main pipeline so the notebook matches the analysis.
fancy_verbs = [
    "delve",
    "underscore",
    "showcase",
    "enhance",
    "exhibit",
    "garner",
    "align",
    "foster",
]

nlp = spacy.load("en_core_web_sm", disable=["ner", "textcat"])


def first_answer(values) -> str:
    return values[0] if isinstance(values, list) and values else ""


def dependency_entropy(text: str, nlp) -> float:
    doc = nlp(str(text))
    deps = [token.dep_ for token in doc if token.dep_ != "punct"]
    if not deps:
        return 0.0
    counts = Counter(deps)
    total = sum(counts.values())
    return -sum((count / total) * math.log(count / total) for count in counts.values())


def count_fancy_verbs(text: str, nlp, verbs: list[str]) -> dict[str, int]:
    doc = nlp(str(text))
    lemmas = [token.lemma_.lower() for token in doc]
    counts = {}
    for verb in verbs:
        counts[verb] = lemmas.count(verb)
    return counts


# Build one dataframe with the relevant fields.
mage = pd.DataFrame({
    "text": ds["text"],
    "label": ds["label"],
    "src": ds["src"],
})
mage["word_count"] = mage["text"].astype(str).str.split().str.len()
mage["dependency_entropy"] = [dependency_entropy(text, nlp) for text in mage["text"]]

verb_counts = pd.DataFrame(
    [count_fancy_verbs(text, nlp, fancy_verbs) for text in mage["text"]],
    index=mage.index,
)
verb_counts.columns = [f"verb_{verb}" for verb in verb_counts.columns]
mage = pd.concat([mage, verb_counts], axis=1)
mage["fancy_verb_count"] = mage[[f"verb_{verb}" for verb in fancy_verbs]].sum(axis=1)
mage["fancy_verb_per_1k_words"] = mage["fancy_verb_count"] / mage["word_count"].replace(0, pd.NA) * 1000.0

# Summary by source.
source_summary = (
    mage.groupby("src")
    .agg(
        n=("fancy_verb_count", "size"),
        mean_fancy_verb_count=("fancy_verb_count", "mean"),
        median_fancy_verb_count=("fancy_verb_count", "median"),
        mean_fancy_verb_per_1k_words=("fancy_verb_per_1k_words", "mean"),
        mean_entropy=("dependency_entropy", "mean"),
        mean_word_count=("word_count", "mean"),
    )
    .reset_index()
    .sort_values("mean_fancy_verb_per_1k_words", ascending=False)
)
print("Source summary")
print(source_summary)

# Summary by label.
label_summary = (
    mage.groupby("label")
    .agg(
        n=("fancy_verb_count", "size"),
        mean_fancy_verb_count=("fancy_verb_count", "mean"),
        median_fancy_verb_count=("fancy_verb_count", "median"),
        mean_fancy_verb_per_1k_words=("fancy_verb_per_1k_words", "mean"),
        mean_entropy=("dependency_entropy", "mean"),
        mean_word_count=("word_count", "mean"),
    )
    .reset_index()
)
print("\nLabel summary")
print(label_summary)

# Per-verb breakdown by source, normalized by document length.
per_verb_source = (
    mage.groupby("src")[[f"verb_{verb}" for verb in fancy_verbs]]
    .mean()
    .sort_index()
)
print("\nPer-verb mean counts by source")
print(per_verb_source)

# Plot the group-level comparison and the per-verb pattern.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

source_order = source_summary["src"].tolist()
axes[0].boxplot(
    [mage.loc[mage["src"] == src, "fancy_verb_per_1k_words"].dropna() for src in source_order],
    labels=source_order,
    showmeans=True,
)
axes[0].set_title("Fancy verbs per 1k words by source")
axes[0].set_ylabel("Fancy verbs per 1k words")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(alpha=0.25, axis="y")

im = axes[1].imshow(per_verb_source.values, aspect="auto", cmap="Blues")
axes[1].set_xticks(range(len(fancy_verbs)))
axes[1].set_xticklabels(fancy_verbs, rotation=30, ha="right")
axes[1].set_yticks(range(len(per_verb_source.index)))
axes[1].set_yticklabels(per_verb_source.index)
axes[1].set_title("Per-verb mean counts by source")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("\nInterpretation check:")
print("If source groups differ on fancy-verb rate, that gives you the same kind of lexical signal as the arXiv marker-verb trends.")
